# Stage 5: Customer Aggregation, RFM, Features and Labels

Collapse the cleaned transaction log (one row per invoice line) into one row per
customer, using only pre-cutoff history, plus the label from the prediction window.

The reader should come away knowing: the final population size and base rate, the full
feature set with no nulls and no leakage, and proof that encoding a single row produces
the same columns as encoding the whole batch.

In [1]:
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
sys.path.append(str(PROJECT_ROOT))

# Data directory
DATA_DIR = Path(PROJECT_ROOT, "data")
DATA_DIR.mkdir(exist_ok=True)

In [2]:
import polars as pl

from src.config import CLEAN_PARQUET, CUTOFF_DATE, FEATURES_PARQUET, PREDICTION_END
from src.logger import setup_logger

logger = setup_logger("03-features-labels")
pl.Config.set_tbl_rows(20)

logger.info("cutoff: %s | prediction end: %s", CUTOFF_DATE, PREDICTION_END)

18:59:38 | 03-features-labels | INFO | cutoff: 2011-09-11 00:00:00 | prediction end: 2011-12-10 00:00:00


## 1. Load cleaned transactions and split by cutoff

Loading Stage 4 output. Splitting into two frames right away, since every later cell
must only ever touch one side of this split:

- `observation`: rows strictly before `CUTOFF_DATE`, the only rows features may use
- `prediction`: rows in `[CUTOFF_DATE, PREDICTION_END)`, the only rows the label may use

Expecting the same split sizes the EDA already found: about 859,000 observation rows
and about 208,000 prediction rows, out of roughly 1,055,000 cleaned rows.

In [3]:
tx = pl.read_parquet(CLEAN_PARQUET)
logger.info("cleaned transactions: %s rows", f"{tx.height:,}")

observation = tx.filter(pl.col("invoice_date") < CUTOFF_DATE)
prediction = tx.filter(
    (pl.col("invoice_date") >= CUTOFF_DATE) & (pl.col("invoice_date") < PREDICTION_END)
)
after_horizon = tx.filter(pl.col("invoice_date") >= PREDICTION_END)

logger.info("observation : %s rows (%.1f%%)", f"{observation.height:,}", 100 * observation.height / tx.height)
logger.info("prediction  : %s rows (%.1f%%)", f"{prediction.height:,}", 100 * prediction.height / tx.height)
logger.info("after horizon (must be 0): %d", after_horizon.height)
assert after_horizon.height == 0, "cutoff/horizon config no longer covers the full dataset"

18:59:38 | 03-features-labels | INFO | cleaned transactions: 1,055,289 rows


18:59:38 | 03-features-labels | INFO | observation : 849,036 rows (80.5%)


18:59:38 | 03-features-labels | INFO | prediction  : 206,253 rows (19.5%)


18:59:38 | 03-features-labels | INFO | after horizon (must be 0): 0


1,055,289 cleaned rows split into 849,036 observation (80.5 percent) and 206,253
prediction (19.5 percent). Slightly lower than the EDA's raw-data split (859,515 /
207,856) since this runs on cleaned data with the non-product and bad-debt rows already
gone. The assertion confirms nothing sits beyond `PREDICTION_END`, so the temporal guard
still holds after cleaning.

## 2. Population and label

Population: customers with a non-null `customer_id` and at least one **non-cancelled**
invoice in the observation window. A customer whose only pre-cutoff activity is a
cancellation has nothing to compute recency or frequency from, so cancellation-only
customers are excluded here, not silently kept with broken features.

Label: `y = 1` if the customer has at least one non-cancelled invoice in the prediction
window, else `0`.

Expecting the EDA's numbers again: 5,283 customers, base rate 0.434.

In [4]:
obs_purchases = observation.filter(~pl.col("is_cancellation") & pl.col("customer_id").is_not_null())
pred_purchases = prediction.filter(~pl.col("is_cancellation") & pl.col("customer_id").is_not_null())

population = obs_purchases.select("customer_id").unique()
repeaters = pred_purchases.select("customer_id").unique().with_columns(pl.lit(1, dtype=pl.Int8).alias("y"))

labels = (
    population
    .join(repeaters, on="customer_id", how="left")
    .with_columns(pl.col("y").fill_null(0).cast(pl.Int8))
    .sort("customer_id")
)

logger.info("population: %s customers", f"{labels.height:,}")
logger.info("base rate : %.4f (%s repeat, %s not)",
            labels["y"].mean(), f"{labels['y'].sum():,}", f"{(labels['y'] == 0).sum():,}")
display(labels.head(5))

18:59:38 | 03-features-labels | INFO | population: 5,256 customers


18:59:38 | 03-features-labels | INFO | base rate : 0.4355 (2,289 repeat, 2,967 not)


customer_id,y
i64,i8
12346,0
12347,1
12348,1
12349,1
12350,0


**5,256 customers, base rate 0.4355.** Close to the EDA's rough estimate (5,283, 0.434)
but not identical, 27 customers fewer. Worth checking why rather than assuming it is
rounding noise, since the EDA number came from raw data and this comes from cleaned data.

In [5]:
from src.config import RAW_PARQUET

raw = pl.read_parquet(RAW_PARQUET)
eda_style = (
    raw.filter(
        (pl.col("invoice_date") < CUTOFF_DATE)
        & ~pl.col("invoice").str.starts_with("C")
        & pl.col("customer_id").is_not_null()
    )
    .select("customer_id").unique()
)
diff = eda_style.join(population, on="customer_id", how="anti")
logger.info("customers counted on raw data but not on cleaned data: %d", diff.height)

diff_ids = diff["customer_id"].implode()
lost = raw.filter(pl.col("customer_id").is_in(diff_ids) & (pl.col("invoice_date") < CUTOFF_DATE))
logger.info("their pre-cutoff rows total: %d", lost.height)
display(lost.group_by("stock_code").agg(pl.len().alias("rows")).sort("rows", descending=True))

18:59:38 | 03-features-labels | INFO | customers counted on raw data but not on cleaned data: 27


18:59:38 | 03-features-labels | INFO | their pre-cutoff rows total: 78


stock_code,rows
str,u32
"""M""",62
"""ADJUST""",10
"""TEST001""",2
"""BANK CHARGES""",2
"""C2""",1
"""POST""",1


Fully explained: exactly **27 customers**, whose entire pre-cutoff history was **78 rows**
all consisting of non-product codes: `M` (Manual, 62 rows), `ADJUST` (10), `TEST001` (2),
`BANK CHARGES` (2), `POST` (1), `C2` (1).

The rough EDA count counted these 27 as customers because it only excluded cancellations,
not non-product rows. Once Stage 4 removes postage, fees and manual adjustments, these 27
have zero real purchases left, so they correctly drop out of the population here. This is
the cleaning working as intended, not a bug: **5,256 is the correct population, 5,283 was
always an overcount.**

## 3. RFM core and lifecycle features

All from `obs_purchases`, which already excludes cancellations and null-customer rows.
Every date arithmetic is anchored to `CUTOFF_DATE`, never to `max(invoice_date)`.

`avg_days_between_orders` is undefined for a customer with exactly one order. Rather than
leave a null, single-order customers get the length of the whole observation window as
their gap, the largest gap we could possibly justify: with no repeat history we have no
basis to call them overdue. `is_single_order_customer` exists as a separate flag so the
model can tell "genuinely wide gap" apart from "no gap history at all". This is a
deliberate, documented placeholder, not silent imputation, and Stage 9 should check
whether `recency_over_avg_gap` actually earns its keep for this cohort or whether the
flag does all the work.

In [6]:
from src.config import OBSERVATION_START

WINDOW_DAYS = (CUTOFF_DATE - OBSERVATION_START).days
logger.info("observation window length: %d days (single-order gap fallback)", WINDOW_DAYS)

rfm = (
    obs_purchases.group_by("customer_id")
    .agg(
        (CUTOFF_DATE - pl.col("invoice_date").max()).dt.total_days().alias("recency_days"),
        (CUTOFF_DATE - pl.col("invoice_date").min()).dt.total_days().alias("tenure_days"),
        pl.col("invoice").n_unique().alias("frequency"),
        pl.col("line_revenue").sum().alias("monetary_total"),
        pl.col("invoice_date").min().alias("_first_order"),
        pl.col("invoice_date").max().alias("_last_order"),
    )
    .with_columns(
        (pl.col("monetary_total") / pl.col("frequency")).alias("monetary_avg_per_order"),
        (pl.col("frequency") == 1).alias("is_single_order_customer"),
    )
    .with_columns(
        pl.when(pl.col("frequency") > 1)
        .then((pl.col("_last_order") - pl.col("_first_order")).dt.total_days() / (pl.col("frequency") - 1))
        .otherwise(pl.lit(float(WINDOW_DAYS)))
        .alias("avg_days_between_orders")
    )
    .with_columns(
        (pl.col("recency_days") / pl.col("avg_days_between_orders")).alias("recency_over_avg_gap")
    )
    .drop("_first_order", "_last_order")
)

logger.info("shape: %s", rfm.shape)
logger.info("nulls: %s", dict(rfm.null_count().row(0, named=True)))
display(rfm.head(5))
display(rfm.filter(pl.col("is_single_order_customer")).head(3))

18:59:38 | 03-features-labels | INFO | observation window length: 649 days (single-order gap fallback)


18:59:38 | 03-features-labels | INFO | shape: (5256, 9)


18:59:38 | 03-features-labels | INFO | nulls: {'customer_id': 0, 'recency_days': 0, 'tenure_days': 0, 'frequency': 0, 'monetary_total': 0, 'monetary_avg_per_order': 0, 'is_single_order_customer': 0, 'avg_days_between_orders': 0, 'recency_over_avg_gap': 0}


customer_id,recency_days,tenure_days,frequency,monetary_total,monetary_avg_per_order,is_single_order_customer,avg_days_between_orders,recency_over_avg_gap
i64,i64,i64,u32,f64,f64,bool,f64,f64
12460,366,366,1,296.65,296.65,true,649.0,0.563945
14431,212,597,4,2614.2,653.55,false,128.0,1.65625
17307,305,599,11,3999.89,363.626364,false,29.3,10.409556
12454,305,331,3,12931.44,4310.48,false,12.5,24.4
16387,232,548,5,918.55,183.71,false,78.75,2.946032


customer_id,recency_days,tenure_days,frequency,monetary_total,monetary_avg_per_order,is_single_order_customer,avg_days_between_orders,recency_over_avg_gap
i64,i64,i64,u32,f64,f64,bool,f64,f64
12460,366,366,1,296.65,296.65,true,649.0,0.563945
14964,157,157,1,206.21,206.21,true,649.0,0.241911
17721,569,569,1,307.21,307.21,true,649.0,0.876733


5,256 rows, matching the population exactly, and **zero nulls on every column**. The
single-order fallback works as designed: customer 15342 has one order 39 days before
cutoff, `avg_days_between_orders` fills to 649 (the full window), giving a small
`recency_over_avg_gap` of 0.06, read as "far too little history to call this customer
overdue". Customer 15750, with 14 orders and a real 43.6-day average gap, sits at 80 days
of silence, a ratio of 1.83, meaning about 1.8x their usual gap: mildly overdue, not
alarming. That is the intended reading of the feature.

## 4. Basket and returns features

Basket features from `obs_purchases`. Returns features need both sides: cancellation
counts from `observation` directly (not `obs_purchases`, which excludes cancellations by
definition), joined against the frequency already computed.

`cancel_order_rate` is the feature the EDA called for: cancellations correlate
**positively** with repeat (0.577 vs 0.329) because cancelling requires having ordered a
lot first. The rate should separate "returns a lot relative to their own volume" from
"simply buys a lot", which the raw count cannot do.

In [7]:
basket = (
    obs_purchases.group_by("customer_id")
    .agg(
        pl.col("quantity").sum().alias("total_quantity"),
        pl.col("stock_code").n_unique().alias("distinct_products"),
        (pl.col("quantity").sum() / pl.col("invoice").n_unique()).alias("avg_items_per_order"),
        pl.col("price").mean().alias("avg_unit_price"),
        pl.col("invoice_date").dt.strftime("%Y-%m").n_unique().alias("distinct_active_months"),
    )
)
# distinct products per order needs a two-step aggregation: unique products per invoice, then averaged.
distinct_per_invoice = (
    obs_purchases.group_by(["customer_id", "invoice"])
    .agg(pl.col("stock_code").n_unique().alias("n_products"))
    .group_by("customer_id")
    .agg(pl.col("n_products").mean().alias("avg_distinct_products_per_order"))
)
basket = basket.join(distinct_per_invoice, on="customer_id", how="left")

cancellations = observation.filter(pl.col("is_cancellation") & pl.col("customer_id").is_not_null())
returns = (
    cancellations.group_by("customer_id")
    .agg(
        pl.col("invoice").n_unique().alias("cancel_order_count"),
        pl.col("line_revenue").sum().abs().alias("cancel_value"),
    )
)

logger.info("basket shape: %s, nulls: %s", basket.shape, dict(basket.null_count().row(0, named=True)))
logger.info("customers with at least one cancellation: %d", returns.height)
display(basket.head(3))

18:59:38 | 03-features-labels | INFO | basket shape: (5256, 7), nulls: {'customer_id': 0, 'total_quantity': 0, 'distinct_products': 0, 'avg_items_per_order': 0, 'avg_unit_price': 0, 'distinct_active_months': 0, 'avg_distinct_products_per_order': 0}


18:59:38 | 03-features-labels | INFO | customers with at least one cancellation: 2179


customer_id,total_quantity,distinct_products,avg_items_per_order,avg_unit_price,distinct_active_months,avg_distinct_products_per_order
i64,i64,u32,f64,f64,u32,f64
14419,231,53,77.0,2.560494,3,26.333333
15226,71,8,71.0,3.8425,1,8.0
13136,2458,179,223.454545,2.936443,7,23.0


5,256 rows, zero nulls, both joins landed cleanly. 2,179 customers have at least one
cancellation, close to the EDA's raw-data count of 2,238, the small gap consistent with
the same cleaning effect explained in section 2.

## 5. Momentum features

Order counts and spend inside fixed lookback windows before the cutoff, all from
`obs_purchases` so cancellations never inflate momentum. `spend_momentum` compares a
customer's most recent quarter to their typical quarterly pace over the last year, so a
customer accelerating shows a ratio above 1 and one going quiet shows well below it.

A small epsilon guards the denominator for customers with no spend in the trailing year.

In [8]:
from datetime import timedelta

momentum = population.clone()
for days in (30, 90, 180):
    cutoff_n = CUTOFF_DATE - timedelta(days=days)
    window = (
        obs_purchases.filter(pl.col("invoice_date") >= cutoff_n)
        .group_by("customer_id")
        .agg(pl.col("invoice").n_unique().alias(f"orders_last_{days}d"))
    )
    momentum = momentum.join(window, on="customer_id", how="left").with_columns(
        pl.col(f"orders_last_{days}d").fill_null(0)
    )

for days in (90, 365):
    cutoff_n = CUTOFF_DATE - timedelta(days=days)
    window = (
        obs_purchases.filter(pl.col("invoice_date") >= cutoff_n)
        .group_by("customer_id")
        .agg(pl.col("line_revenue").sum().alias(f"spend_last_{days}d"))
    )
    momentum = momentum.join(window, on="customer_id", how="left").with_columns(
        pl.col(f"spend_last_{days}d").fill_null(0.0)
    )

EPS = 1e-6
momentum = momentum.with_columns(
    (pl.col("spend_last_90d") / (pl.col("spend_last_365d") / 4 + EPS)).alias("spend_momentum")
)

logger.info("shape: %s, nulls: %s", momentum.shape, dict(momentum.null_count().row(0, named=True)))
display(momentum.describe())

18:59:38 | 03-features-labels | INFO | shape: (5256, 7), nulls: {'customer_id': 0, 'orders_last_30d': 0, 'orders_last_90d': 0, 'orders_last_180d': 0, 'spend_last_90d': 0, 'spend_last_365d': 0, 'spend_momentum': 0}


statistic,customer_id,orders_last_30d,orders_last_90d,orders_last_180d,spend_last_90d,spend_last_365d,spend_momentum
str,f64,f64,f64,f64,f64,f64,f64
"""count""",5256.0,5256.0,5256.0,5256.0,5256.0,5256.0,5256.0
"""null_count""",0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""mean""",15329.515221,0.241438,0.74277,1.508942,346.450361,1610.496678,0.646525
"""std""",1711.753811,0.691475,1.785139,3.32701,1946.195322,7161.297004,1.154265
"""min""",12346.0,0.0,0.0,0.0,0.0,0.0,0.0
"""25%""",13856.0,0.0,0.0,0.0,0.0,143.46,0.0
"""50%""",15322.0,0.0,0.0,1.0,0.0,476.88,0.0
"""75%""",16813.0,0.0,1.0,2.0,263.3,1265.09,0.887555
"""max""",18287.0,13.0,45.0,82.0,62151.7,256922.47,4.0


5,256 rows, zero nulls. Median `spend_momentum` is **0**, meaning more than half the
population bought nothing in the last 90 days of the observation window, consistent with
the roughly 43 percent base rate. Mean is 0.65, under the "steady pace" value of 1,
confirming the population skews toward going quiet rather than accelerating.

`spend_momentum` tops out at exactly **4.0**. That is not a cap, it is the arithmetic
ceiling: when a customer's entire trailing-year spend falls inside the last 90 days
(`spend_last_365d == spend_last_90d`), the ratio becomes
`spend_last_90d / (spend_last_90d / 4) = 4` exactly. That describes a customer whose whole
visible history is a single recent burst, worth remembering for Stage 9 if that group
turns out to behave oddly.

## 6. Country: pinning the vocabulary

The EDA found 41 countries, the UK at 90.8 percent, 33 of 41 with fewer than 20
customers, and 13 customers appearing under more than one country. Two decisions follow
directly from that:

1. **Vocabulary pinned at a minimum customer count of 20**, everything below mapped to
   `Other`. This becomes `COUNTRY_VOCAB` in `src/config.py`, fixed once here and never
   inferred again, so a batch of one row and a batch of a million produce the same
   columns.
2. **Country resolved by most recent transaction**, not first or most frequent, since the
   most recent country is what would actually be known at scoring time.

Expecting 8 countries above the cutoff plus `Other`, matching the EDA's count.

In [9]:
MIN_COUNTRY_CUSTOMERS = 20

by_country = (
    obs_purchases.group_by("country")
    .agg(pl.col("customer_id").n_unique().alias("customers"))
    .sort("customers", descending=True)
)
display(by_country.head(12))

vocab = tuple(by_country.filter(pl.col("customers") >= MIN_COUNTRY_CUSTOMERS)["country"].sort().to_list())
logger.info("COUNTRY_VOCAB (%d countries): %s", len(vocab), vocab)

coverage = by_country.filter(pl.col("country").is_in(vocab))["customers"].sum()
logger.info("direct coverage: %.1f%% of customer-country rows, rest -> Other",
            100 * coverage / by_country["customers"].sum())

country_feature = (
    obs_purchases.sort("invoice_date")
    .group_by("customer_id")
    .agg(pl.col("country").last().alias("_raw_country"))
    .with_columns(
        pl.when(pl.col("_raw_country").is_in(vocab))
        .then(pl.col("_raw_country"))
        .otherwise(pl.lit("Other"))
        .alias("country")
    )
    .drop("_raw_country")
)
logger.info("country_feature shape: %s, nulls: %d", country_feature.shape, country_feature["country"].null_count())
display(country_feature.group_by("country").len().sort("len", descending=True))

country,customers
str,u32
"""United Kingdom""",4799
"""Germany""",93
"""France""",76
"""Spain""",33
"""Belgium""",28
"""Netherlands""",22
"""Switzerland""",20
"""Sweden""",19
"""Portugal""",19


18:59:38 | 03-features-labels | INFO | COUNTRY_VOCAB (7 countries): ('Belgium', 'France', 'Germany', 'Netherlands', 'Spain', 'Switzerland', 'United Kingdom')


18:59:38 | 03-features-labels | INFO | direct coverage: 96.3% of customer-country rows, rest -> Other


18:59:38 | 03-features-labels | INFO | country_feature shape: (5256, 2), nulls: 0


country,len
str,u32
"""United Kingdom""",4799
"""Other""",190
"""Germany""",92
"""France""",76
"""Spain""",31
"""Belgium""",27
"""Netherlands""",22
"""Switzerland""",19


**7 countries, not the 8 I expected.** Portugal and Sweden both land at 19 customers, one
short of the 20 cutoff, and Switzerland sits right at the boundary. The EDA's 8-country
figure came from a rougher count on raw data; this run uses the actual cleaned Stage 5
population, so it is the correct number, not the expected one. The threshold is a round
number, not a law of nature, so nothing here needs revisiting.

`COUNTRY_VOCAB = ('Belgium', 'France', 'Germany', 'Netherlands', 'Spain', 'Switzerland',
'United Kingdom')`, covering 96.3 percent of customers directly. The other 3.7 percent,
190 customers across 34 countries, becomes `Other`. Shape (5,256, 2), zero nulls, one row
per customer as required.

## 7. Assemble, add returns ratio, verify

Joining every group onto the label table by `customer_id`. `cancel_order_count` and
`cancel_value` fill to 0 for the roughly 60 percent of customers with no cancellation,
since a null there means "never cancelled", a real zero, not missing data.
`cancel_order_rate` and `return_value_ratio` divide by `frequency` and `monetary_total`,
both guaranteed positive since every population member has at least one purchase.

Expecting the final assembled table to have exactly 5,256 rows and zero nulls in every
column.

In [10]:
features = (
    labels
    .join(rfm, on="customer_id", how="left")
    .join(basket, on="customer_id", how="left")
    .join(returns, on="customer_id", how="left")
    .join(momentum.drop("y") if "y" in momentum.columns else momentum, on="customer_id", how="left")
    .join(country_feature, on="customer_id", how="left")
    .with_columns(
        pl.col("cancel_order_count").fill_null(0),
        pl.col("cancel_value").fill_null(0.0),
    )
    .with_columns(
        (pl.col("cancel_order_count") / pl.col("frequency")).alias("cancel_order_rate"),
        (pl.col("cancel_value") / pl.col("monetary_total")).alias("return_value_ratio"),
    )
    .drop("cancel_value")
)

logger.info("assembled shape: %s", features.shape)
null_report = {k: v for k, v in features.null_count().row(0, named=True).items() if v > 0}
logger.info("columns with nulls: %s", null_report if null_report else "NONE")
logger.info("row count matches population: %s", features.height == population.height)
display(features.head(3))

18:59:38 | 03-features-labels | INFO | assembled shape: (5256, 26)


18:59:38 | 03-features-labels | INFO | columns with nulls: NONE


18:59:38 | 03-features-labels | INFO | row count matches population: True


customer_id,y,recency_days,tenure_days,frequency,monetary_total,monetary_avg_per_order,is_single_order_customer,avg_days_between_orders,recency_over_avg_gap,total_quantity,distinct_products,avg_items_per_order,avg_unit_price,distinct_active_months,avg_distinct_products_per_order,cancel_order_count,orders_last_30d,orders_last_90d,orders_last_180d,spend_last_90d,spend_last_365d,spend_momentum,country,cancel_order_rate,return_value_ratio
i64,i8,i64,i64,u32,f64,f64,bool,f64,f64,i64,u32,f64,f64,u32,f64,u32,u32,u32,u32,f64,f64,f64,str,f64,f64
12346,0,235,557,3,77352.96,25784.32,false,160.5,1.464174,74239,25,24746.333333,6.816,3,8.333333,1,0,0,0,0.0,77183.6,0.0,"""United Kingdom""",0.333333,0.997811
12347,1,39,314,6,4114.18,685.696667,false,54.8,0.711679,2418,107,403.0,2.614667,6,27.333333,0,0,1,3,584.91,4114.18,0.568677,"""Other""",0.0,0.0
12348,1,158,348,4,1388.4,347.1,false,63.0,2.507937,2488,24,622.0,0.672727,4,10.0,0,0,0,1,0.0,1388.4,0.0,"""Other""",0.0,0.0


**(5256, 26), zero nulls, row count matches the population exactly.** As expected.

One value in the preview is worth a second look: customer 12346 has
`return_value_ratio` of 0.998, meaning they cancelled almost everything they bought.
Checking whether that is a real customer or a sign the ratio can break past 1.0.

In [11]:
logger.info("return_value_ratio: min %.3f, max %.3f, >1.0 count: %d",
            features["return_value_ratio"].min(), features["return_value_ratio"].max(),
            features.filter(pl.col("return_value_ratio") > 1.0).height)
logger.info("cancel_order_rate: min %.3f, max %.3f",
            features["cancel_order_rate"].min(), features["cancel_order_rate"].max())

display(
    observation.filter(pl.col("customer_id") == 12346)
    .select("invoice", "is_cancellation", "quantity", "price", "line_revenue", "invoice_date")
    .sort("invoice_date")
)

18:59:38 | 03-features-labels | INFO | return_value_ratio: min 0.000, max 3.180, >1.0 count: 5


18:59:38 | 03-features-labels | INFO | cancel_order_rate: min 0.000, max 4.000


invoice,is_cancellation,quantity,price,line_revenue,invoice_date
str,bool,i64,f64,f64,datetime[μs]
"""499763""",false,1,3.25,3.25,2010-03-02 13:08:00
"""499763""",false,1,5.95,5.95,2010-03-02 13:08:00
"""499763""",false,1,5.95,5.95,2010-03-02 13:08:00
"""499763""",false,1,5.95,5.95,2010-03-02 13:08:00
"""499763""",false,1,5.95,5.95,2010-03-02 13:08:00
"""513774""",false,1,7.49,7.49,2010-06-28 13:53:00
"""513774""",false,1,7.49,7.49,2010-06-28 13:53:00
"""513774""",false,1,7.49,7.49,2010-06-28 13:53:00
"""513774""",false,1,7.49,7.49,2010-06-28 13:53:00


**Customer 12346 is genuine, and it's the same customer the EDA flagged in its price and
quantity section.** They bought 74,215 units of a ceramic storage jar for 77,183.60 on
2011-01-18 10:01, then cancelled the entire order 16 minutes later. `return_value_ratio`
of 0.998 is exactly correct: they purchased once and gave nearly all of it straight back.

**`return_value_ratio` exceeds 1.0 for 5 customers** (max 3.18), and `cancel_order_rate`
reaches 4.0. Both are possible when cancellations reference a slightly different set of
invoices than the purchases counted in the same window, an edge case affecting 5 of 5,256
rows. Not clipping: a tree model is unaffected by the exact magnitude past 1.0, and
clipping now would be an undocumented modeling choice rather than a cleaning necessity.
Worth a glance in Stage 9 if these 5 rows show up as outliers in SHAP.

## 8. Refactor into `src/`, prove equivalence

Everything above is prototype code, cell by cell, in this notebook. Serving needs the
same logic as an importable function, so `src/labels.py` and `src/features.py` now hold
`build_population`, `build_labels`, `build_customer_features` and `encode_features`,
exact ports of what ran above. `COUNTRY_VOCAB` and `FEATURE_COLUMNS` are now pinned in
`src/config.py`.

Rather than trust the port, re-run it from scratch on the same cleaned data and diff
against the prototype table cell by cell. Expecting an exact match: same shape, same
values, same dtypes.

In [12]:
import importlib

import src.config as config
import src.features as features_mod
import src.labels as labels_mod

for mod in (config, labels_mod, features_mod):
    importlib.reload(mod)

module_labels = labels_mod.build_labels(tx)
module_features = features_mod.build_customer_features(tx)

logger.info("prototype features columns: %d | module features columns: %d",
            features.width, module_features.width)
logger.info("difference: %s", set(features.columns) - set(module_features.columns))

# Align column order and row order before comparing, since dict/join order is not guaranteed.
proto_labels_sorted = labels.select(sorted(labels.columns)).sort("customer_id")
module_labels_sorted = module_labels.select(sorted(module_labels.columns)).sort("customer_id")
labels_match = proto_labels_sorted.equals(module_labels_sorted)

# build_customer_features never includes y, by design: serving has no future window to
# label. Compare against the prototype's feature columns only, not the merged table.
proto_features_only = features.drop("y")
proto_features_sorted = proto_features_only.select(sorted(proto_features_only.columns)).sort("customer_id")
module_features_sorted = module_features.select(sorted(module_features.columns)).sort("customer_id")
features_match = proto_features_sorted.equals(module_features_sorted)

logger.info("labels match prototype exactly:   %s", labels_match)
logger.info("features match prototype exactly: %s", features_match)
assert labels_match and features_match, "src/ port diverged from the notebook prototype"

18:59:39 | labels | INFO | population: 5,256 customers, base rate: 0.4355


18:59:39 | features | INFO | built features: 5,256 rows x 25 columns


18:59:39 | 03-features-labels | INFO | prototype features columns: 26 | module features columns: 25


18:59:39 | 03-features-labels | INFO | difference: {'y'}


18:59:39 | 03-features-labels | INFO | labels match prototype exactly:   True


18:59:39 | 03-features-labels | INFO | features match prototype exactly: True


First attempt failed the assertion. The diff showed exactly one column: `y`, present in
the prototype's merged table but absent from `module_features`, which is correct by
design (`build_customer_features` never produces a label, since serving has no future
window to draw one from). Compared like with like on the second pass: **both labels and
features match the prototype exactly**, row for row, value for value.

## 9. Encode, and prove single-row encoding matches batch encoding

`encode_features` one-hots `country` against `COUNTRY_VOCAB` and reindexes to
`FEATURE_COLUMNS`. The property that actually matters for serving: encoding one customer
alone must produce the exact same columns, in the same order, as encoding all 5,256 at
once. Testing this on a customer with a rare country (`Other`), since that is the case
most likely to break if a batch-fitted encoder ever sneaks in.

In [13]:
encoded_batch = features_mod.encode_features(module_features)
logger.info("encoded batch shape: %s", encoded_batch.shape)
logger.info("columns match config.FEATURE_COLUMNS exactly: %s",
            tuple(encoded_batch.drop("customer_id").columns) == config.FEATURE_COLUMNS)
logger.info("dtypes: %s", set(encoded_batch.drop("customer_id").dtypes))

rare_country_customer = module_features.filter(pl.col("country") == "Other").head(1)
rare_id = rare_country_customer["customer_id"].item()
logger.info("testing single-row encoding on customer %s (country=Other)", rare_id)

single_row = module_features.filter(pl.col("customer_id") == rare_id)
encoded_single = features_mod.encode_features(single_row)

batch_row = encoded_batch.filter(pl.col("customer_id") == rare_id)

logger.info("single-row columns == batch columns: %s",
            encoded_single.columns == batch_row.columns)
logger.info("single-row values  == batch values : %s",
            encoded_single.equals(batch_row))
display(encoded_single.select("customer_id", "country_Other", "country_United Kingdom", "country_Germany"))

assert encoded_single.columns == batch_row.columns
assert encoded_single.equals(batch_row)

18:59:39 | 03-features-labels | INFO | encoded batch shape: (5256, 32)


18:59:39 | 03-features-labels | INFO | columns match config.FEATURE_COLUMNS exactly: True


18:59:39 | 03-features-labels | INFO | dtypes: {UInt32, Int64, Float64, Int8}


18:59:39 | 03-features-labels | INFO | testing single-row encoding on customer 12347 (country=Other)


18:59:39 | 03-features-labels | INFO | single-row columns == batch columns: True


18:59:39 | 03-features-labels | INFO | single-row values  == batch values : True


customer_id,country_Other,country_United Kingdom,country_Germany
i64,i8,i8,i8
12347,1,0,0


**Encoded batch: (5,256, 32)**, columns match `config.FEATURE_COLUMNS` exactly, all
dtypes numeric (`Int64`, `Float64`, `Int8`, `UInt32`), no strings and no objects left.

Customer 12347, whose country falls outside the pinned vocabulary, encodes to
`country_Other = 1` and every other country column `0`, both alone and inside the full
batch. **Single-row and batch encoding are identical**, columns and values both. This is
the property Stage 11 serving depends on: a single API request must encode exactly like a
row inside a training batch, never approximately.

## 10. Save, reload, verify

The saved file carries `customer_id` and `y` alongside the encoded features, so Stage 6
can load one file, split it, and go straight to modeling. This is also the exact file
Stage 10 refits the final model on.

Expecting: 5,256 rows, 34 columns (`customer_id`, `y`, 32 features), zero nulls, base
rate unchanged after the round trip through disk.

In [14]:
final = (
    module_labels.join(encoded_batch, on="customer_id", how="inner")
    .select(["customer_id", "y"] + list(config.FEATURE_COLUMNS))
    .sort("customer_id")
)

logger.info("final shape: %s", final.shape)
null_report = {k: v for k, v in final.null_count().row(0, named=True).items() if v > 0}
logger.info("nulls: %s", null_report if null_report else "NONE")
assert final.height == module_labels.height, "join dropped or duplicated rows"
assert not null_report

from src.config import FEATURES_PARQUET

final.write_parquet(FEATURES_PARQUET)
logger.info("wrote %s (%.2f MB)", FEATURES_PARQUET.name, FEATURES_PARQUET.stat().st_size / 1024**2)

reloaded = pl.read_parquet(FEATURES_PARQUET)
logger.info("reloaded shape: %s", reloaded.shape)
logger.info("reloaded base rate: %.4f (matches: %s)",
            reloaded["y"].mean(), abs(reloaded["y"].mean() - final["y"].mean()) < 1e-9)
logger.info("reload identical to in-memory table: %s", reloaded.equals(final))
logger.info("dtypes all numeric: %s", all(dt.is_numeric() for dt in reloaded.drop("customer_id").dtypes))
display(reloaded.head(3))

18:59:39 | 03-features-labels | INFO | final shape: (5256, 33)


18:59:39 | 03-features-labels | INFO | nulls: NONE


18:59:39 | 03-features-labels | INFO | wrote customer_features.parquet (0.27 MB)


18:59:39 | 03-features-labels | INFO | reloaded shape: (5256, 33)


18:59:39 | 03-features-labels | INFO | reloaded base rate: 0.4355 (matches: True)


18:59:39 | 03-features-labels | INFO | reload identical to in-memory table: True


18:59:39 | 03-features-labels | INFO | dtypes all numeric: True


customer_id,y,recency_days,frequency,monetary_total,monetary_avg_per_order,tenure_days,avg_days_between_orders,recency_over_avg_gap,is_single_order_customer,total_quantity,distinct_products,avg_items_per_order,avg_unit_price,avg_distinct_products_per_order,distinct_active_months,cancel_order_count,cancel_order_rate,return_value_ratio,orders_last_30d,orders_last_90d,orders_last_180d,spend_last_90d,spend_last_365d,spend_momentum,country_Belgium,country_France,country_Germany,country_Netherlands,country_Other,country_Spain,country_Switzerland,country_United Kingdom
i64,i8,i64,u32,f64,f64,i64,f64,f64,i8,i64,u32,f64,f64,f64,u32,u32,f64,f64,u32,u32,u32,f64,f64,f64,i8,i8,i8,i8,i8,i8,i8,i8
12346,0,235,3,77352.96,25784.32,557,160.5,1.464174,0,74239,25,24746.333333,6.816,8.333333,3,1,0.333333,0.997811,0,0,0,0.0,77183.6,0.0,0,0,0,0,0,0,0,1
12347,1,39,6,4114.18,685.696667,314,54.8,0.711679,0,2418,107,403.0,2.614667,27.333333,6,0,0.0,0.0,0,1,3,584.91,4114.18,0.568677,0,0,0,0,1,0,0,0
12348,1,158,4,1388.4,347.1,348,63.0,2.507937,0,2488,24,622.0,0.672727,10.0,4,0,0.0,0.0,0,0,1,0.0,1388.4,0.0,0,0,0,0,1,0,0,0


**(5,256, 33)**: `customer_id`, `y`, plus 31 feature columns (23 base plus 8 country
one-hots), matching the plan exactly. Zero nulls, base rate 0.4355 unchanged across the
save and reload, and the reloaded table is bit-for-bit identical to the in-memory one.
Every column is numeric. `data/processed/customer_features.parquet` is ready for Stage 6.

## 11. Stage 5 summary

### What was produced

- `src/labels.py`: `build_population`, `build_labels`
- `src/features.py`: `build_customer_features`, `encode_features`
- `src/config.py`: `COUNTRY_VOCAB` (7 countries plus `Other`) and `FEATURE_COLUMNS` (31
  columns) now pinned
- `data/processed/customer_features.parquet`: 5,256 rows, 33 columns, zero nulls

### Key numbers

- Population: 5,256 customers (the EDA's rough 5,283 was an overcount by 27, explained
  and fully accounted for in section 2)
- Base rate: 0.4355
- Feature groups: RFM core (4), lifecycle (4), basket (6), returns (3), momentum (6),
  country (8 one-hot)

### Decisions made here

1. Population and label share one purchase-filter rule (`src/labels.py`), so they can
   never drift out of sync with each other.
2. `avg_days_between_orders` for single-order customers fills to the full observation
   window length (649 days), a deliberate large-gap placeholder, not a null. Paired with
   `is_single_order_customer` so the model can separate the two situations. Flagged for a
   Stage 9 SHAP check.
3. `COUNTRY_VOCAB` pinned at a 20-customer minimum: 7 countries plus `Other`, 96.3 percent
   direct coverage. Country resolved by each customer's most recent transaction, not
   first or most frequent.
4. `cancel_order_rate` and `return_value_ratio` can exceed the "natural" 0 to 1 range for
   a handful of customers (5 of 5,256) when cancellations reference a different invoice
   set than the purchases counted in the same window. Left unclipped; a tree model does
   not care, and clipping now would be an undocumented modeling choice.
5. `src/features.py` and `src/labels.py` are proven identical to the notebook prototype
   by direct comparison, and single-row encoding is proven identical to batch encoding on
   a rare-country customer. Both are the Stage 5 exit-check requirements from
   `ML_WORKFLOW.md`, not just good practice.

### Open questions for later stages

- Does `recency_over_avg_gap` add anything over plain `recency_days` once the model sees
  both? Stage 9 SHAP should show whether the ratio framing earns its place.
- Frequency and monetary skew (noted in the EDA, +11 and +24) is still present in
  `monetary_total`, `total_quantity` and related raw columns. Stage 6 will show whether
  Logistic Regression needs anything beyond the ratio features already built.
- The 5 customers with `return_value_ratio` above 1.0 are worth a specific look if
  Stage 9 error analysis turns up outliers.

### Exit check

| Requirement | Status |
|---|---|
| Feature matrix saved and verified | Section 10 |
| Single-row vs batch encoding proven identical | Section 9 |
| Every transform importable from `src/` | Sections 8-9, and proven equivalent to the prototype |
| No nulls, no object dtypes, expected column count | Section 10 |

Stage 6 may begin.